# Manual Search Playground\n\nRun queries against the full search pipeline and inspect per-channel breakdowns.

In [1]:
# ── Setup: imports, env, and connections ──────────────────────────────────────
import asyncio
import os
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

# Ensure the project root is on sys.path so "db.*" and "implementation.*" resolve
_project_root = str(Path.cwd().parent) if Path.cwd().name == "api" else str(Path.cwd())
if _project_root not in sys.path:
    sys.path.insert(0, _project_root)

from dotenv import load_dotenv
load_dotenv(Path(_project_root) / ".env")

# Open Postgres pool and Redis before any DB calls
from db.postgres import pool, fetch_movie_cards
from db.redis import init_redis

await pool.open()
await init_redis()

# Qdrant client (created at import time, no explicit open needed)
from db.qdrant import qdrant_client

from movie_ingestion.tracker import TRACKER_DB_PATH

# Search components
from db.lexical_search import lexical_search
from db.vector_search import run_vector_search
from db.vector_scoring import calculate_vector_scores, RELEVANCE_RAW_WEIGHTS
from db.metadata_scoring import create_metadata_scores
from db.reranking import rerank_candidates
from db.search import SearchCandidate, _timed_channel_weights, _timed_metadata_preferences, _correct_channel_weights

from db.trending_movies import refresh_trending_scores
from db.redis import init_redis, close_redis

from implementation.classes.schemas import MetadataFilters
from implementation.classes.enums import RelevanceSize, ReceptionType, VectorName

print("All connections initialized.")

/Users/michaelkeohane/Documents/movie-finder-rag/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All connections initialized.


In [40]:
init_redis()
await refresh_trending_scores()
close_redis()

/var/folders/2r/_4wt88q52lb5vr94n449kl5w0000gn/T/ipykernel_36913/953718965.py:1: RuntimeWarning: coroutine 'init_redis' was never awaited
  init_redis()


pages_needed: 25


TMDB returned 499 movies, fewer than requested 500 — scoring against actual count


<coroutine object close_redis at 0x14a35abc0>

In [2]:
# ── Query input ───────────────────────────────────────────────────────────────
# Edit these values and re-run from here downward.

QUERY = "iconic twist ending"

# Hard filters — set any to None to disable
FILTERS = MetadataFilters(
    min_release_ts=None,       # unix timestamp, e.g. int(datetime(2000,1,1).timestamp())
    max_release_ts=None,
    min_runtime=None,          # minutes
    max_runtime=None,
    min_maturity_rank=None,
    max_maturity_rank=None,
    genres=None,               # list of Genre enum values
    audio_languages=None,      # list of Language enum values
    watch_offer_keys=None,     # list of int watch provider keys
)

print(f"Query: {QUERY!r}")
print(f"Filters active: {FILTERS.is_active}")

Query: 'iconic twist ending'
Filters active: False


In [3]:
# ── Run search (capturing all intermediates) ───────────────────��──────────────
start = time.perf_counter()

# Phase A — Fire all 4 LLM + retrieval tasks in parallel
channel_weights_task = asyncio.create_task(_timed_channel_weights(QUERY))

lexical_result, vector_result, metadata_pref_timed = await asyncio.gather(
    lexical_search(QUERY, FILTERS),
    run_vector_search(
        query=QUERY,
        metadata_filters=FILTERS,
        qdrant_client=qdrant_client,
    ),
    _timed_metadata_preferences(QUERY),
)
metadata_preferences, metadata_pref_ms = metadata_pref_timed

# Phase B — Score vector candidates, merge with lexical
scoring_result = calculate_vector_scores(vector_result)
final_vector_scores = scoring_result.final_scores

merged: dict[int, SearchCandidate] = {}
for movie_id, v_score in final_vector_scores.items():
    merged[movie_id] = SearchCandidate(movie_id=movie_id, vector_score=v_score, lexical_score=0.0)
for lc in lexical_result.candidates:
    if lc.movie_id in merged:
        merged[lc.movie_id].lexical_score = lc.normalized_lexical_score
    else:
        merged[lc.movie_id] = SearchCandidate(movie_id=lc.movie_id, vector_score=0.0, lexical_score=lc.normalized_lexical_score)

# Phase C — Metadata scoring
candidates_list = list(merged.values())
reception_scores: dict[int, float | None] = {}
candidates_list = await create_metadata_scores(metadata_preferences, candidates_list, reception_scores)

# Phase D — Channel weights
channel_weights_result, channel_weights_ms = await channel_weights_task

has_lexical_entities = len(lexical_result.debug.extracted_entities.entity_candidates) > 0
has_active_metadata = metadata_preferences.has_active_preferences()

if channel_weights_result is not None:
    vector_rel, lexical_rel, metadata_rel = _correct_channel_weights(
        vector_relevance=RelevanceSize(channel_weights_result.vector_relevance),
        lexical_relevance=RelevanceSize(channel_weights_result.lexical_relevance),
        metadata_relevance=RelevanceSize(channel_weights_result.metadata_relevance),
        has_lexical_entities=has_lexical_entities,
        has_active_metadata=has_active_metadata,
    )
    raw_v = RELEVANCE_RAW_WEIGHTS[vector_rel]
    raw_l = RELEVANCE_RAW_WEIGHTS[lexical_rel]
    raw_m = RELEVANCE_RAW_WEIGHTS[metadata_rel]
    total = raw_v + raw_l + raw_m
    if total > 0:
        w_vector, w_lexical, w_metadata = raw_v / total, raw_l / total, raw_m / total
    else:
        w_vector = w_lexical = w_metadata = 1.0 / 3.0
else:
    w_vector = w_lexical = w_metadata = 1.0 / 3.0

for c in candidates_list:
    c.final_score = w_vector * c.vector_score + w_lexical * c.lexical_score + w_metadata * c.metadata_score

# Phase E — Quality prior reranking
from db.postgres import fetch_reception_scores
raw_reception_type = metadata_preferences.reception_preference.reception_type
reception_type = raw_reception_type if isinstance(raw_reception_type, ReceptionType) else ReceptionType(raw_reception_type)
if not reception_scores and candidates_list:
    reception_scores = await fetch_reception_scores([c.movie_id for c in candidates_list])
rerank_candidates(candidates_list, reception_scores, reception_type)

elapsed_ms = (time.perf_counter() - start) * 1000

# ── Timing breakdown ─��───────────────────────────────────────────────────────
print(f"Search complete: {len(candidates_list)} candidates in {elapsed_ms:.0f}ms\n")
print("TIMING BREAKDOWN")
print("-" * 60)

# Lexical channel
lex_dbg = lexical_result.debug
print(f"  Lexical search total:          {lex_dbg.latency_ms:>8.0f}ms")
print(f"    LLM entity extraction:       {lex_dbg.llm_generation_time_ms:>8.0f}ms")

# Metadata preferences
print(f"  Metadata preferences LLM:      {metadata_pref_ms:>8.0f}ms")

# Channel weights
print(f"  Channel weights LLM:           {channel_weights_ms:>8.0f}ms")

# Vector search
vec_dbg = vector_result.debug
if vec_dbg:
    print(f"  Vector search wall clock:      {vec_dbg.wall_clock_ms:>8.0f}ms  ({vec_dbg.total_jobs_executed} jobs, {vec_dbg.total_candidates} candidates)")
    for job in vec_dbg.per_job_stats:
        llm_str = f"{job.llm_generation_time_ms:.0f}ms" if job.llm_generation_time_ms is not None else "n/a"
        print(f"    {job.score_field:<40s}  embed={job.embedding_time_ms:.0f}ms  llm={llm_str:<6s}  qdrant={job.latency_ms:.0f}ms  ({job.candidates_returned} hits)")

print(f"\n  Total end-to-end:              {elapsed_ms:>8.0f}ms")

Subquery LLM returned relevant_subquery_text as None for watch_context, skipping subquery search.
Weight LLM returned not_relevant for plot_analysis, skipping original-query search.
Subquery LLM returned relevant_subquery_text as None for plot_events, skipping subquery search.
Subquery LLM returned relevant_subquery_text as None for viewer_experience, skipping subquery search.
Weight LLM returned not_relevant for watch_context, skipping original-query search.
Subquery LLM returned relevant_subquery_text as None for production, skipping subquery search.
Weight LLM returned not_relevant for production, skipping original-query search.
Weight LLM returned not_relevant for viewer_experience, skipping original-query search.
Vector search complete: 7 jobs, 2602 unique candidates, 42562.03ms wall clock
Search complete: 2602 candidates in 44197ms

TIMING BREAKDOWN
------------------------------------------------------------
  Lexical search total:              3062ms
    LLM entity extraction: 

In [7]:
# ── Helper: bulk resolve movie_id → (title, year, overview) ──────────────────
import sqlite3

# TRACKER_DB_PATH is relative (ingestion_data/tracker.db), so resolve it
# against the project root to work regardless of the kernel's cwd.
_tracker_db_abs = Path(_project_root) / TRACKER_DB_PATH

async def _resolve_titles(movie_ids: list[int]) -> dict[int, tuple[str, int | None]]:
    """Returns {movie_id: (title, release_year)} from Postgres movie_card."""
    if not movie_ids:
        return {}
    cards = await fetch_movie_cards(movie_ids)
    result = {}
    for card in cards:
        mid = card["movie_id"]
        title = card["title"] or f"[id:{mid}]"
        release_ts = card.get("release_ts")
        year = datetime.fromtimestamp(release_ts, tz=timezone.utc).year if release_ts else None
        result[mid] = (title, year)
    return result

def _resolve_overviews(movie_ids: list[int]) -> dict[int, str]:
    """Returns {movie_id: overview} from the tracker SQLite imdb_data table."""
    if not movie_ids:
        return {}
    conn = sqlite3.connect(str(_tracker_db_abs))
    try:
        placeholders = ",".join("?" for _ in movie_ids)
        rows = conn.execute(
            f"SELECT tmdb_id, overview FROM imdb_data WHERE tmdb_id IN ({placeholders})",
            movie_ids,
        ).fetchall()
        return {row[0]: row[1] for row in rows if row[1]}
    finally:
        conn.close()

def _fmt(title: str, year: int | None) -> str:
    return f"{title} ({year})" if year else title

# Collect every movie_id we'll need to display
all_display_ids = list({c.movie_id for c in candidates_list})
title_map = await _resolve_titles(all_display_ids)

# Only fetch overviews for the top 25 (no need for all 10k)
top_25_ids = [c.movie_id for c in candidates_list[:25]]
overview_map = _resolve_overviews(top_25_ids)
print(f"Resolved {len(title_map)} titles, {len(overview_map)} overviews")

Resolved 2602 titles, 25 overviews


In [44]:
# ── Top 25 results ────────────────────────────────────────────────────────────
print("=" * 70)
print(f"TOP 25 RESULTS — {QUERY!r}")
print("=" * 70)
for i, c in enumerate(candidates_list[:25], 1):
    title, year = title_map.get(c.movie_id, (f"[id:{c.movie_id}]", None))
    overview = overview_map.get(c.movie_id, "")
    print(f"  {i:>2}. {c.final_score:.4f}  {_fmt(title, year)}")
    if overview:
        # Truncate long overviews to keep output scannable
        display = (overview[:200] + "...") if len(overview) > 200 else overview
        print(f"      {display}")
print()

TOP 25 RESULTS — 'trending'
   1. 0.4274  I Swear (2025)
      John Davidson: diagnosed with Tourette's syndrome at a young age which alienated him from his peers, he struggled with a condition few people had witnessed.
   2. 0.1389  Prince Harry: Being the Spare (2023)
      If the Palace thought that 2023 would be a quieter year after the drama and tragedy of 2022 then the release of Prince Harry's autobiography Spare has just about destroyed all hope. The book has cause...
   3. 0.1294  Don't Trust the Quiet Ones (2026)
      After an influencer is murdered at a mall meet and greet, a divorced mom reports what she saw. When threats follow, her ex takes them to a remote cabin, but the real danger may be closer than she thin...
   4. 0.1153  Breaking a Monster (2015)
      BREAKING A MONSTER chronicles the break-out year of the band UNLOCKING THE TRUTH, following 13-year-old members Alec Atkins, Malcolm Brickhouse and Jarad Dawkins as they first encounter stardom and th...
   5. 0.110

In [4]:
# ── Channel weights ───────────────────────────────────────────────────────────
print("=" * 70)
print("CHANNEL WEIGHTS")
print("=" * 70)
if channel_weights_result:
    print(f"  LLM raw:     vector={channel_weights_result.vector_relevance}  "
          f"lexical={channel_weights_result.lexical_relevance}  "
          f"metadata={channel_weights_result.metadata_relevance}")
    print(f"  Corrected:   vector={vector_rel.value}  lexical={lexical_rel.value}  metadata={metadata_rel.value}")
print(f"  Final weights: vector={w_vector:.3f}  lexical={w_lexical:.3f}  metadata={w_metadata:.3f}")
print()

CHANNEL WEIGHTS
  LLM raw:     vector=large  lexical=not_relevant  metadata=not_relevant
  Corrected:   vector=large  lexical=not_relevant  metadata=small
  Final weights: vector=0.750  lexical=0.000  metadata=0.250



In [35]:
# ── Lexical breakdown ─────────────────────────────────────────────────────────
print("=" * 70)
print("LEXICAL ENTITIES EXTRACTED")
print("=" * 70)
entities = lexical_result.debug.extracted_entities.entity_candidates
if entities:
    for e in entities:
        excl = " [EXCLUDE]" if e.exclude_from_results else ""
        print(f"  {e.most_likely_category}: {e.corrected_and_normalized_entity!r}{excl}")
else:
    print("  (none)")

print()
print("TOP 10 LEXICAL MATCHES")
print("-" * 50)
for i, lc in enumerate(lexical_result.candidates[:10], 1):
    title, year = title_map.get(lc.movie_id, (f"[id:{lc.movie_id}]", None))
    print(f"  {i:>2}. {lc.normalized_lexical_score:.4f}  {_fmt(title, year)}")
if not lexical_result.candidates:
    print("  (no lexical matches)")
print()

LEXICAL ENTITIES EXTRACTED
  (none)

TOP 10 LEXICAL MATCHES
--------------------------------------------------
  (no lexical matches)



In [5]:
# ── Metadata preferences breakdown ────────────────────────────────────────────
print("=" * 70)
print("METADATA PREFERENCES EXTRACTED")
print("=" * 70)
prefs = metadata_preferences
pref_fields = [
    ("Release date",    prefs.release_date_preference),
    ("Duration",        prefs.duration_preference),
    ("Genres",          prefs.genres_preference),
    ("Audio languages", prefs.audio_languages_preference),
    ("Watch providers", prefs.watch_providers_preference),
    ("Maturity rating", prefs.maturity_rating_preference),
    ("Popular/trending",prefs.popular_trending_preference),
    ("Reception",       prefs.reception_preference),
    ("Budget size",     prefs.budget_size_preference),
]
for name, pref in pref_fields:
    print(f"  {name}: {pref}")

print()
print("TOP 10 METADATA MATCHES")
print("-" * 50)
meta_sorted = sorted(candidates_list, key=lambda c: c.metadata_score, reverse=True)
for i, c in enumerate(meta_sorted[:10], 1):
    title, year = title_map.get(c.movie_id, (f"[id:{c.movie_id}]", None))
    print(f"  {i:>2}. {c.metadata_score:.4f}  {_fmt(title, year)}")
print()

METADATA PREFERENCES EXTRACTED
  Release date: result=None
  Duration: result=None
  Genres: result=None
  Audio languages: result=None
  Watch providers: result=None
  Maturity rating: result=None
  Popular/trending: prefers_trending_movies=False prefers_popular_movies=True
  Reception: reception_type='no_preference'
  Budget size: budget_size='no_preference'

TOP 10 METADATA MATCHES
--------------------------------------------------


NameError: name 'title_map' is not defined

In [9]:
# ── Per-vector-space breakdown ─────────────────────────────────────────────────
print("=" * 70)
print("VECTOR SEARCH — PER-SPACE BREAKDOWN")
print("=" * 70)

# Map VectorName → subquery text from the search result
subqueries = vector_result.vector_subqueries
subquery_map = {
    VectorName.PLOT_EVENTS:          subqueries.plot_events_subquery,
    VectorName.PLOT_ANALYSIS:        subqueries.plot_analysis_subquery,
    VectorName.VIEWER_EXPERIENCE:    subqueries.viewer_experience_subquery,
    VectorName.WATCH_CONTEXT:        subqueries.watch_context_subquery,
    VectorName.NARRATIVE_TECHNIQUES: subqueries.narrative_techniques_subquery,
    VectorName.PRODUCTION:           subqueries.production_subquery,
    VectorName.RECEPTION:            subqueries.reception_subquery,
}

# Space contexts contain the normalized weights from the scoring pipeline
context_by_name = {ctx.name: ctx for ctx in scoring_result.space_contexts}

# Per-space normalized scores (from VectorScoringResult debug output)
per_space = scoring_result.per_space_normalized

print(subquery_map)

for space_name in VectorName:
    ctx = context_by_name.get(space_name)
    weight = ctx.normalized_weight if ctx else 0.0
    active = ctx.is_active if ctx else False

    print(f"\n  ── {space_name.value} ──")
    print(f"  Weight: {weight:.4f}  |  Active: {active}")

    # Show subquery (anchor has none)
    if space_name == VectorName.ANCHOR:
        print(f"  Subquery: (uses original query)")
    else:
        sq = subquery_map.get(space_name)
        print(f"  Subquery: {sq!r}" if sq else "  Subquery: (none — space not relevant)")

    # Top 10 candidates in this space by normalized score
    space_scores = per_space.get(space_name, {})
    if space_scores:
        top_10 = sorted(space_scores.items(), key=lambda x: x[1], reverse=True)[:10]
        for rank, (mid, score) in enumerate(top_10, 1):
            title, year = title_map.get(mid, (f"[id:{mid}]", None))
            print(f"    {rank:>2}. {score:.4f}  {_fmt(title, year)}")
    else:
        print("    (no candidates)")

print()

VECTOR SEARCH — PER-SPACE BREAKDOWN
{<VectorName.PLOT_EVENTS: 'plot_events'>: None, <VectorName.PLOT_ANALYSIS: 'plot_analysis'>: "hidden truth revealed, things aren't what they seem, deception, perception vs reality, unreliable truth, revelation changes everything, identity revealed, truth concealed, master manipulation, moral ambiguity, psychological thriller, mystery, revelation, subverted expectations", <VectorName.VIEWER_EXPERIENCE: 'viewer_experience'>: None, <VectorName.WATCH_CONTEXT: 'watch_context'>: None, <VectorName.NARRATIVE_TECHNIQUES: 'narrative_techniques'>: "plot twist / reversal, twist ending, slow-burn reveal, misdirection editing, unreliable narrator, Chekhov's gun, dramatic irony", <VectorName.PRODUCTION: 'production'>: None, <VectorName.RECEPTION: 'reception'>: 'inventive structure, mind-blowing twist, shocking reveal, iconic twist, unforgettable ending, rewatchable, keeps you guessing, clever plotting, bold narrative choices, praised twist'}

  ── anchor ──
  Weigh